<a href="https://colab.research.google.com/github/cepulmano/ai-healthcare-medicine/blob/main/day3/ClinicalNamedEntityRecognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clinical Named-Entity Recognition

In [3]:
!pip install medspacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.6/244.6 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.9/69.9 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.4/568.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.5/408.5 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.8 MB/s eta 0:00:00
  Created wheel for medspacy: filename=medspacy-1.3.1-py3-none-any.whl size=313704 sha256=61dba4167f0fba8be769

In [6]:
import medspacy
from medspacy.ner import TargetRule
from medspacy.visualization import visualize_ent

# Load medspacy model
nlp = medspacy.load()
print(nlp.pipe_names)

clinical_text = """
Past Medical History:
1. Atrial fibrillation
2. Type II Diabetes Mellitus

Assessment and Plan:
There is no evidence of pneumonia.
Continue warfarin for Afib.
Follow up for management of type 2 DM.
Rule out hypertension.
"""

# Add rules for target concept extraction
target_matcher = nlp.get_pipe("medspacy_target_matcher")
target_rules = [
    TargetRule("atrial fibrillation", "PROBLEM"),
    TargetRule("atrial fibrillation", "PROBLEM", pattern=[{"LOWER": "afib"}]),
    TargetRule("pneumonia", "PROBLEM"),
    TargetRule("hypertension", "PROBLEM"),
    TargetRule("Type II Diabetes Mellitus", "PROBLEM",
              pattern=[
                  {"LOWER": "type"},
                  {"LOWER": {"IN": ["2", "ii", "two"]}},
                  {"LOWER": {"IN": ["dm", "diabetes"]}},
                  {"LOWER": "mellitus", "OP": "?"}
              ]),
    TargetRule("warfarin", "MEDICATION")
]
target_matcher.add(target_rules)

doc = nlp(clinical_text)

print("--- Extracted Clinical Entities ---")
for ent in doc.ents:
    status = "Negated" if ent._.is_negated else "Affirmed"
    if ent._.is_uncertain: status = "Speculation"
    print(f"Entity: {ent.text:<30} | Label: {ent.label_:<10} | {status:<15}")

visualize_ent(doc)

2026-07-22 02:32:24.099 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 1 'Past' marked as sentence start (span begin)
2026-07-22 02:32:24.099 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 5 '
' marked as sentence start (span end whitespace)
2026-07-22 02:32:24.101 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] GAP DETECTED: tokens 5-5 (idx 22-22) between spans 22-23
2026-07-22 02:32:24.102 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 5 '
' marked as sentence start (whitespace in gap between spans)
2026-07-22 02:32:24.104 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 6 '1' marked as sentence start (span begin)
2026-07-22 02:32:24.105 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 10 '
' m

['medspacy_pyrush', 'medspacy_target_matcher', 'medspacy_context']
--- Extracted Clinical Entities ---
Entity: Atrial fibrillation            | Label: PROBLEM    | Affirmed       
Entity: Type II Diabetes Mellitus      | Label: PROBLEM    | Affirmed       
Entity: pneumonia                      | Label: PROBLEM    | Negated        
Entity: warfarin                       | Label: MEDICATION | Affirmed       
Entity: Afib                           | Label: PROBLEM    | Affirmed       
Entity: type 2 DM                      | Label: PROBLEM    | Affirmed       
Entity: hypertension                   | Label: PROBLEM    | Speculation    
